In [80]:
# ============================================================
# GENERATE THREE ADDRESS CODE USING FLEX AND BISON
# Google Colab - Single Cell
# ============================================================

# 1. Install FLEX, BISON and GCC
!apt-get update -qq
!apt-get install -y flex bison gcc -qq


# ============================================================
# 2. Create tac.l
# ============================================================

with open("tac.l", "w") as f:
    f.write(r'''
%{
#include "tac.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%option noyywrap

%%

[a-zA-Z][a-zA-Z0-9]* {
    yylval.str = strdup(yytext);
    return ID;
}

[0-9]+ {
    yylval.str = strdup(yytext);
    return NUM;
}

[ \t\n]+ {
    /* Ignore spaces and newlines */
}

. {
    return yytext[0];
}

%%
''')


# ============================================================
# 3. Create tac.y
# ============================================================

with open("tac.y", "w") as f:
    f.write(r'''
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int tempCount = 1;

int yylex(void);
int yyerror(char *s);
%}

%union {
    char *str;
}

%token <str> ID NUM
%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt:
    ID '=' expr
    {
        printf("%s = %s\n", $1, $3);
    }
    ;

expr:
      expr '+' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printf("%s = %s + %s\n", temp, $1, $3);
          $$ = strdup(temp);
      }

    | expr '-' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printf("%s = %s - %s\n", temp, $1, $3);
          $$ = strdup(temp);
      }

    | expr '*' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printf("%s = %s * %s\n", temp, $1, $3);
          $$ = strdup(temp);
      }

    | expr '/' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printf("%s = %s / %s\n", temp, $1, $3);
          $$ = strdup(temp);
      }

    | '(' expr ')'
      {
          $$ = $2;
      }

    | ID
      {
          $$ = $1;
      }

    | NUM
      {
          $$ = $1;
      }
    ;

%%

int main()
{
    printf("Enter the expression:\n");
    yyparse();
    return 0;
}

int yyerror(char *s)
{
    printf("Error: %s\n", s);
    return 0;
}
''')


# ============================================================
# 4. Remove old generated files
# ============================================================

!rm -f tac.tab.c tac.tab.h lex.yy.c tac


# ============================================================
# 5. Generate BISON and FLEX files
# ============================================================

!bison -d tac.y
!flex tac.l


# ============================================================
# 6. Compile
# ============================================================

!gcc tac.tab.c lex.yy.c -o tac -lfl


# ============================================================
# 7. Give input
# ============================================================

with open("input.txt", "w") as f:
    f.write("a = b + c * d\n")


# ============================================================
# 8. Execute program
# ============================================================

import subprocess

result = subprocess.run(
    ["./tac"],
    stdin=open("input.txt", "r"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Enter the expression:
t1 = c * d
t2 = b + t1
a = t2

